# 05 Interpretation and Lookalike PoC

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
from IPython.display import display

ARTIFACT_DIR = Path.cwd() / "notebook_artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)


In [ ]:
# Validierungs- und Robustheitsartefakte aus dem vorherigen Notebook laden

with open(ARTIFACT_DIR / "04_validation_robustness_artifacts.pkl", "rb") as f:
    validation_robustness_artifacts = pickle.load(f)

debitor_features_final = validation_robustness_artifacts["debitor_features_final"]
robustness_selected_models = validation_robustness_artifacts["robustness_selected_models"]
robustness_validation_df = validation_robustness_artifacts["robustness_validation_df"]
robustness_cluster_sizes_overview_df = validation_robustness_artifacts["robustness_cluster_sizes_overview_df"]

print("Geladen:", ARTIFACT_DIR / "04_validation_robustness_artifacts.pkl")
print("Verfügbare Robustheitsmodelle:", list(robustness_selected_models.keys()))

## Inhaltliche Segmentbewertung und Ableitung von Clusterprofilen
Nach der Robustheitsprüfung werden die inhaltlich relevanten Varianten `B_Groessenreduziert` und `C_Relational` vertieft analysiert. Ziel ist es, die Cluster nicht nur anhand der im Modell verwendeten Merkmale, sondern über zusätzliche business-relevante Kennzahlen zu beschreiben und daraus erste fachliche Segmentprofile abzuleiten.

In [837]:
# Relevante Varianten für die inhaltliche Segmentbewertung festlegen

interpretation_variants = ["B_Groessenreduziert", "C_Relational"]

business_profile_features = [
    "invoice_count",
    "active_years_count",
    "recency_days",
    "net_index_sum",
    "net_per_invoice",
    "placement_count",
    "placement_rate",
    "leadership_rate",
    "invoices_per_active_year",
    "net_per_active_year"
]

available_business_profile_features = [
    col for col in business_profile_features if col in debitor_features_final.columns
]

print("Für die inhaltliche Interpretation verwendete Zusatzmerkmale:")
print(available_business_profile_features)

Für die inhaltliche Interpretation verwendete Zusatzmerkmale:
['invoice_count', 'active_years_count', 'recency_days', 'net_index_sum', 'net_per_invoice', 'placement_count', 'placement_rate', 'leadership_rate', 'invoices_per_active_year', 'net_per_active_year']


In [838]:
# Erweiterte Clusterprofile für die inhaltlich relevanten Varianten erzeugen

interpretation_profiles = {}

for variant_name in interpretation_variants:
    if variant_name not in robustness_selected_models:
        print(f"Variante {variant_name} wurde nicht in robustness_selected_models gefunden.")
        continue

    model = robustness_selected_models[variant_name]
    result_df = model["result_df"].copy()
    cluster_col = model["cluster_col"]

    extended_profile_df = (
        result_df.groupby(cluster_col)[available_business_profile_features]
        .mean()
        .round(3)
    )

    cluster_size_df = (
        result_df[cluster_col]
        .value_counts()
        .sort_index()
        .reset_index()
    )
    cluster_size_df.columns = ["cluster", "count"]

    interpretation_profiles[variant_name] = {
        "cluster_col": cluster_col,
        "cluster_sizes": cluster_size_df,
        "extended_profile_df": extended_profile_df
    }

for variant_name, profile_data in interpretation_profiles.items():
    print("=" * 100)
    print(f"Variante: {variant_name}")
    print()
    print("Clustergrößen:")
    display(profile_data["cluster_sizes"])
    print()
    print("Erweiterte Clusterprofile:")
    display(profile_data["extended_profile_df"])
    print()

Variante: B_Groessenreduziert

Clustergrößen:


,cluster,count
0,0,197
1,1,364
2,2,213
3,3,587



Erweiterte Clusterprofile:


,invoice_count,active_years_count,recency_days,net_index_sum,net_per_invoice,placement_count,placement_rate,leadership_rate,invoices_per_active_year,net_per_active_year
cluster_B_Groessenreduziert,,,,,,,,,,
0,5.411,1.751,2851.269,0.677,0.136,2.640,0.554,0.069,2.923,0.376
1,6.484,1.764,828.827,0.695,0.113,1.335,0.181,0.808,3.582,0.389
2,37.911,5.869,1042.535,2.965,0.083,10.146,0.269,0.162,5.940,0.473
3,6.218,1.722,2992.407,0.399,0.063,1.211,0.140,0.019,3.361,0.211



Variante: C_Relational

Clustergrößen:


,cluster,count
0,0,196
1,1,595
2,2,358
3,3,212



Erweiterte Clusterprofile:


,invoice_count,active_years_count,recency_days,net_index_sum,net_per_invoice,placement_count,placement_rate,leadership_rate,invoices_per_active_year,net_per_active_year
cluster_C_Relational,,,,,,,,,,
0,4.745,1.883,2785.929,0.577,0.134,2.255,0.547,0.072,2.479,0.312
1,5.718,1.934,2928.156,0.364,0.063,1.108,0.142,0.019,2.882,0.177
2,5.955,1.953,859.704,0.606,0.109,1.201,0.180,0.797,3.024,0.313
3,40.976,4.854,1160.033,3.327,0.093,11.071,0.273,0.200,8.652,0.762


In [839]:
# Clusterprofile innerhalb jeder Variante relativ zum jeweiligen Variantenmittel vergleichen

interpretation_relative_profiles = {}

for variant_name in interpretation_variants:
    if variant_name not in interpretation_profiles:
        continue

    model = robustness_selected_models[variant_name]
    result_df = model["result_df"].copy()
    cluster_col = model["cluster_col"]

    cluster_means = result_df.groupby(cluster_col)[available_business_profile_features].mean()
    overall_means = result_df[available_business_profile_features].mean()

    relative_profile_df = (cluster_means / overall_means).round(2)

    interpretation_relative_profiles[variant_name] = relative_profile_df

for variant_name, relative_df in interpretation_relative_profiles.items():
    print("=" * 100)
    print(f"Relative Clusterprofile im Vergleich zum Variantenmittel: {variant_name}")
    display(relative_df)
    print()

Relative Clusterprofile im Vergleich zum Variantenmittel: B_Groessenreduziert


,invoice_count,active_years_count,recency_days,net_index_sum,net_per_invoice,placement_count,placement_rate,leadership_rate,invoices_per_active_year,net_per_active_year
cluster_B_Groessenreduziert,,,,,,,,,,
0,0.49,0.73,1.37,0.74,1.51,0.93,2.40,0.27,0.78,1.16
1,0.58,0.74,0.40,0.76,1.26,0.47,0.78,3.11,0.95,1.20
2,3.41,2.46,0.50,3.22,0.92,3.56,1.16,0.62,1.58,1.46
3,0.56,0.72,1.43,0.43,0.70,0.43,0.61,0.07,0.89,0.65



Relative Clusterprofile im Vergleich zum Variantenmittel: C_Relational


,invoice_count,active_years_count,recency_days,net_index_sum,net_per_invoice,placement_count,placement_rate,leadership_rate,invoices_per_active_year,net_per_active_year
cluster_C_Relational,,,,,,,,,,
0,0.43,0.79,1.33,0.63,1.49,0.79,2.37,0.28,0.66,0.97
1,0.51,0.81,1.40,0.40,0.70,0.39,0.62,0.07,0.77,0.55
2,0.53,0.82,0.41,0.66,1.21,0.42,0.78,3.07,0.80,0.97
3,3.68,2.03,0.56,3.62,1.03,3.89,1.18,0.77,2.30,2.35


### Methodische Arbeitslogik zur Segmentbenennung

Die Segmentbenennung erfolgt nicht rein auf Basis der Metriken, sondern anhand einer kombinierten Betrachtung aus Aktualität, Beziehungsdauer, wirtschaftlicher Intensität, Placement-Fokus und Führungsorientierung. Ziel ist eine nachvollziehbare Business-Interpretation, nicht die rein technische Beschreibung von Clusterunterschieden.

In [840]:
# Arbeitsmatrix zur späteren Segmentbenennung vorbereiten

segment_naming_summary = []

for variant_name in interpretation_variants:
    if variant_name not in interpretation_profiles:
        continue

    abs_df = interpretation_profiles[variant_name]["extended_profile_df"]
    rel_df = interpretation_relative_profiles[variant_name]

    for cluster_id in abs_df.index:
        row = {
            "variante": variant_name,
            "cluster": int(cluster_id)
        }

        for col in available_business_profile_features:
            row[f"{col}_mean"] = round(abs_df.loc[cluster_id, col], 3)
            row[f"{col}_rel"] = round(rel_df.loc[cluster_id, col], 2)

        segment_naming_summary.append(row)

segment_naming_summary_df = pd.DataFrame(segment_naming_summary)
segment_naming_summary_df

,variante,cluster,invoice_count_mean,invoice_count_rel,active_years_count_mean,active_years_count_rel,recency_days_mean,recency_days_rel,net_index_sum_mean,net_index_sum_rel,...,placement_count_mean,placement_count_rel,placement_rate_mean,placement_rate_rel,leadership_rate_mean,leadership_rate_rel,invoices_per_active_year_mean,invoices_per_active_year_rel,net_per_active_year_mean,net_per_active_year_rel
0,B_Groessenreduziert,0,5.411,0.49,1.751,0.73,2851.269,1.37,0.677,0.74,...,2.640,0.93,0.554,2.40,0.069,0.27,2.923,0.78,0.376,1.16
1,B_Groessenreduziert,1,6.484,0.58,1.764,0.74,828.827,0.40,0.695,0.76,...,1.335,0.47,0.181,0.78,0.808,3.11,3.582,0.95,0.389,1.20
2,B_Groessenreduziert,2,37.911,3.41,5.869,2.46,1042.535,0.50,2.965,3.22,...,10.146,3.56,0.269,1.16,0.162,0.62,5.940,1.58,0.473,1.46
3,B_Groessenreduziert,3,6.218,0.56,1.722,0.72,2992.407,1.43,0.399,0.43,...,1.211,0.43,0.140,0.61,0.019,0.07,3.361,0.89,0.211,0.65
4,C_Relational,0,4.745,0.43,1.883,0.79,2785.929,1.33,0.577,0.63,...,2.255,0.79,0.547,2.37,0.072,0.28,2.479,0.66,0.312,0.97
5,C_Relational,1,5.718,0.51,1.934,0.81,2928.156,1.40,0.364,0.40,...,1.108,0.39,0.142,0.62,0.019,0.07,2.882,0.77,0.177,0.55
6,C_Relational,2,5.955,0.53,1.953,0.82,859.704,0.41,0.606,0.66,...,1.201,0.42,0.180,0.78,0.797,3.07,3.024,0.80,0.313,0.97
7,C_Relational,3,40.976,3.68,4.854,2.03,1160.033,0.56,3.327,3.62,...,11.071,3.89,0.273,1.18,0.200,0.77,8.652,2.30,0.762,2.35


## Methodische Auswahl der Interpretationslösung

Die Referenzbaseline liefert die stärksten Clusterqualitätsmetriken, ist jedoch stark unbalanciert und bildet primär eine Größen- und Intensitätslogik ab. Für die inhaltliche Segmentinterpretation wird daher die Variante `C_Relational` als fachliche Hauptlösung verwendet. Diese Variante liefert zwar schwächere Metriken, zeigt jedoch balanciertere und inhaltlich besser interpretierbare Beziehungstypen.

## Inhaltliche Interpretation der Segmente

Auf Basis der relationalen Variantenlogik lassen sich vier vorläufige Kundentypen unterscheiden:

### Cluster 0: Selektive Placement-Mandate
Dieses Segment umfasst Debitoren mit vergleichsweise hohem Wert pro Rechnung und hoher Placement-Rate, jedoch geringer Beziehungsdauer und begrenzter Kontinuität. Es handelt sich damit eher um selektive, wertige Einzel- oder Kurzbeziehungen mit klarem Placement-Fokus.

### Cluster 1: Schwache Gelegenheitskunden
Dieses Segment ist durch geringe wirtschaftliche Intensität, niedrige Placement- und Leadership-Anteile sowie hohe Recency geprägt. Die Beziehungen erscheinen eher randständig, wenig fokussiert und aus Akquise-Sicht nur begrenzt attraktiv.

### Cluster 2: Führungsorientierte Suchkunden
Dieses Segment weist besonders hohe Leadership-Anteile und vergleichsweise aktuelle Aktivität auf. Es bildet damit einen Kundentyp ab, der stark auf Führungspositionen ausgerichtet ist und fachlich gut zum Executive-Search-Kontext passt.

### Cluster 3: Langfristige Kernkunden
Dieses Segment vereint hohe Beziehungsdauer, hohe Aktivität und die stärkste wirtschaftliche Intensität. Es beschreibt damit die strategisch relevantesten und historisch tragfähigsten Kundenbeziehungen im Bestand.

## Erste Business-Einordnung

Aus Akquise-Sicht erscheinen insbesondere zwei Segmente besonders relevant: die langfristigen Kernkunden sowie die führungsorientierten Suchkunden. Selektive Placement-Mandate können als ergänzend attraktiver Spezialtyp betrachtet werden, während schwache Gelegenheitskunden eher als niedrig priorisierte Vergleichsgruppe dienen. Diese Einordnung bildet die Grundlage für den nächsten Schritt, in dem die identifizierten Muster in ein konzeptionelles Lookalike-Prinzip für externe Zielunternehmen überführt werden.

## Heuristischer Lookalike-PoC für externe Zielunternehmen

Nach der internen Segmentierung bestehender Debitoren wird im nächsten Schritt ein kleiner heuristischer Proof of Concept für externe Zielunternehmen aufgebaut. Ziel ist nicht die Entwicklung eines validierten externen Scoring-Modells, sondern die nachvollziehbare Demonstration, wie sich interne Segmentmuster in eine einfache Ähnlichkeitslogik für potenzielle Zielkunden übertragen lassen.

Dafür wird eine kleine externe Beispiel-Unternehmensliste mit fünf Proxy-Signalen aufgebaut:

- `leadership_hiring_signal`
- `hiring_continuity_signal`
- `growth_signal`
- `transformation_signal`
- `management_change_signal`

Die Signale werden auf einer einfachen ordinalen Skala von 0 bis 2 abgebildet:

- `0` = kein erkennbares Signal
- `1` = moderates Signal
- `2` = starkes Signal

Anschließend werden zwei segmentbezogene Fit-Heuristiken angewendet:

- `fit_longterm_core_client`
- `fit_leadership_search_client`

Die folgende Demonstration dient ausschließlich der methodischen Veranschaulichung eines möglichen Lookalike-Transfers.

In [841]:
import pandas as pd

external_companies_poc = pd.DataFrame([
    {
        "company_name": "AlphaTech Systems",
        "industry": "Technology",
        "country": "Germany",
        "source_reference": "PoC example",
        "leadership_hiring_signal": 2,
        "hiring_continuity_signal": 2,
        "growth_signal": 2,
        "transformation_signal": 1,
        "management_change_signal": 1,
        "signal_evidence": "Strong executive hiring activity, sustained recruiting, visible growth dynamics."
    },
    {
        "company_name": "NordIndustrial Group",
        "industry": "Industrial Manufacturing",
        "country": "Germany",
        "source_reference": "PoC example",
        "leadership_hiring_signal": 1,
        "hiring_continuity_signal": 2,
        "growth_signal": 1,
        "transformation_signal": 1,
        "management_change_signal": 0,
        "signal_evidence": "Ongoing hiring demand with moderate growth and selective leadership hiring."
    },
    {
        "company_name": "Crestline Health Solutions",
        "industry": "Healthcare",
        "country": "Switzerland",
        "source_reference": "PoC example",
        "leadership_hiring_signal": 2,
        "hiring_continuity_signal": 1,
        "growth_signal": 1,
        "transformation_signal": 2,
        "management_change_signal": 1,
        "signal_evidence": "Leadership search relevance supported by transformation and organizational change."
    },
    {
        "company_name": "Meridian Finance Advisory",
        "industry": "Financial Services",
        "country": "Germany",
        "source_reference": "PoC example",
        "leadership_hiring_signal": 1,
        "hiring_continuity_signal": 1,
        "growth_signal": 0,
        "transformation_signal": 0,
        "management_change_signal": 1,
        "signal_evidence": "Some executive movement, but weak continuity and no clear growth signal."
    },
    {
        "company_name": "BluePeak Retail Europe",
        "industry": "Retail",
        "country": "Netherlands",
        "source_reference": "PoC example",
        "leadership_hiring_signal": 0,
        "hiring_continuity_signal": 2,
        "growth_signal": 2,
        "transformation_signal": 1,
        "management_change_signal": 0,
        "signal_evidence": "Strong continuity and growth, but no visible leadership hiring signal."
    },
    {
        "company_name": "NovaMobility Ventures",
        "industry": "Mobility",
        "country": "Germany",
        "source_reference": "PoC example",
        "leadership_hiring_signal": 2,
        "hiring_continuity_signal": 1,
        "growth_signal": 2,
        "transformation_signal": 2,
        "management_change_signal": 2,
        "signal_evidence": "Strong leadership demand combined with growth, change and transformation dynamics."
    },
    {
        "company_name": "TradCore Services",
        "industry": "Business Services",
        "country": "Austria",
        "source_reference": "PoC example",
        "leadership_hiring_signal": 0,
        "hiring_continuity_signal": 0,
        "growth_signal": 0,
        "transformation_signal": 0,
        "management_change_signal": 0,
        "signal_evidence": "No clear external signals indicating target similarity."
    },
    {
        "company_name": "Vertex Energy Transition",
        "industry": "Energy",
        "country": "Denmark",
        "source_reference": "PoC example",
        "leadership_hiring_signal": 2,
        "hiring_continuity_signal": 1,
        "growth_signal": 1,
        "transformation_signal": 2,
        "management_change_signal": 2,
        "signal_evidence": "Executive hiring need in a high-transformation environment with clear management change."
    }
])

external_companies_poc

,company_name,industry,country,source_reference,leadership_hiring_signal,hiring_continuity_signal,growth_signal,transformation_signal,management_change_signal,signal_evidence
0,AlphaTech Systems,Technology,Germany,PoC example,2,2,2,1,1,"Strong executive hiring activity, sustained re..."
1,NordIndustrial Group,Industrial Manufacturing,Germany,PoC example,1,2,1,1,0,Ongoing hiring demand with moderate growth and...
2,Crestline Health Solutions,Healthcare,Switzerland,PoC example,2,1,1,2,1,Leadership search relevance supported by trans...
3,Meridian Finance Advisory,Financial Services,Germany,PoC example,1,1,0,0,1,"Some executive movement, but weak continuity a..."
4,BluePeak Retail Europe,Retail,Netherlands,PoC example,0,2,2,1,0,"Strong continuity and growth, but no visible l..."
5,NovaMobility Ventures,Mobility,Germany,PoC example,2,1,2,2,2,"Strong leadership demand combined with growth,..."
6,TradCore Services,Business Services,Austria,PoC example,0,0,0,0,0,No clear external signals indicating target si...
7,Vertex Energy Transition,Energy,Denmark,PoC example,2,1,1,2,2,Executive hiring need in a high-transformation...


### Anwendung der segmentbezogenen Fit-Heuristiken

Im nächsten Schritt werden die zuvor definierten Regelheuristiken auf die externe Beispiel-Unternehmensliste angewendet. Ziel ist es, für jedes Unternehmen eine einfache segmentbezogene Fit-Einschätzung zu erzeugen.

Die Bewertung erfolgt getrennt für zwei priorisierte Zielsegmente:

- `fit_longterm_core_client`
- `fit_leadership_search_client`

Zusätzlich wird je Fit-Feld eine kurze textliche Begründung erzeugt, um die Zuordnung nachvollziehbar zu machen.

In [842]:
def evaluate_longterm_core_client(row):
    hiring = row["hiring_continuity_signal"]
    growth = row["growth_signal"]
    leadership = row["leadership_hiring_signal"]
    transformation = row["transformation_signal"]
    management_change = row["management_change_signal"]

    strategic_support_count = sum([
        leadership >= 1,
        transformation >= 1,
        management_change >= 1
    ])

    if hiring == 0 or growth == 0:
        return "kein_fit", "Kein Fit, da Kontinuität oder Wachstum nicht ausreichend erkennbar sind."

    if hiring >= 2 and growth >= 2 and strategic_support_count >= 1:
        return "hoher_fit", "Hoher Fit durch starke Hiring-Kontinuität, starkes Wachstum und mindestens ein strategisches Verstärkungssignal."

    if hiring >= 1 and growth >= 1 and strategic_support_count >= 1:
        return "guter_fit", "Guter Fit durch vorhandene Kontinuität und Wachstum sowie ergänzende strategische Signale."

    return "mindestfit", "Mindestfit durch erkennbare Kontinuität und Wachstum, jedoch ohne ausreichend starke strategische Verstärkung."


def evaluate_leadership_search_client(row):
    leadership = row["leadership_hiring_signal"]
    transformation = row["transformation_signal"]
    management_change = row["management_change_signal"]

    context_support_count = sum([
        transformation >= 1,
        management_change >= 1
    ])

    if leadership == 0:
        return "kein_fit", "Kein Fit, da kein sichtbares Leadership-Hiring-Signal vorliegt."

    if leadership >= 2 and context_support_count >= 1:
        return "hoher_fit", "Hoher Fit durch starkes Leadership-Hiring-Signal in Verbindung mit Transformations- oder Managementwechsel-Kontext."

    if leadership >= 1 and context_support_count >= 1:
        return "guter_fit", "Guter Fit durch erkennbaren Leadership-Bedarf mit zusätzlichem Veränderungs- oder Wechselkontext."

    return "mindestfit", "Mindestfit durch sichtbares Leadership-Hiring-Signal ohne klaren zusätzlichen Kontext."

In [843]:
external_companies_poc[["fit_longterm_core_client", "fit_reason_longterm"]] = (
    external_companies_poc.apply(
        lambda row: pd.Series(evaluate_longterm_core_client(row)),
        axis=1
    )
)

external_companies_poc[["fit_leadership_search_client", "fit_reason_leadership"]] = (
    external_companies_poc.apply(
        lambda row: pd.Series(evaluate_leadership_search_client(row)),
        axis=1
    )
)

external_companies_poc

,company_name,industry,country,source_reference,leadership_hiring_signal,hiring_continuity_signal,growth_signal,transformation_signal,management_change_signal,signal_evidence,fit_longterm_core_client,fit_reason_longterm,fit_leadership_search_client,fit_reason_leadership
0,AlphaTech Systems,Technology,Germany,PoC example,2,2,2,1,1,"Strong executive hiring activity, sustained re...",hoher_fit,"Hoher Fit durch starke Hiring-Kontinuität, sta...",hoher_fit,Hoher Fit durch starkes Leadership-Hiring-Sign...
1,NordIndustrial Group,Industrial Manufacturing,Germany,PoC example,1,2,1,1,0,Ongoing hiring demand with moderate growth and...,guter_fit,Guter Fit durch vorhandene Kontinuität und Wac...,guter_fit,Guter Fit durch erkennbaren Leadership-Bedarf ...
2,Crestline Health Solutions,Healthcare,Switzerland,PoC example,2,1,1,2,1,Leadership search relevance supported by trans...,guter_fit,Guter Fit durch vorhandene Kontinuität und Wac...,hoher_fit,Hoher Fit durch starkes Leadership-Hiring-Sign...
3,Meridian Finance Advisory,Financial Services,Germany,PoC example,1,1,0,0,1,"Some executive movement, but weak continuity a...",kein_fit,"Kein Fit, da Kontinuität oder Wachstum nicht a...",guter_fit,Guter Fit durch erkennbaren Leadership-Bedarf ...
4,BluePeak Retail Europe,Retail,Netherlands,PoC example,0,2,2,1,0,"Strong continuity and growth, but no visible l...",hoher_fit,"Hoher Fit durch starke Hiring-Kontinuität, sta...",kein_fit,"Kein Fit, da kein sichtbares Leadership-Hiring..."
5,NovaMobility Ventures,Mobility,Germany,PoC example,2,1,2,2,2,"Strong leadership demand combined with growth,...",guter_fit,Guter Fit durch vorhandene Kontinuität und Wac...,hoher_fit,Hoher Fit durch starkes Leadership-Hiring-Sign...
6,TradCore Services,Business Services,Austria,PoC example,0,0,0,0,0,No clear external signals indicating target si...,kein_fit,"Kein Fit, da Kontinuität oder Wachstum nicht a...",kein_fit,"Kein Fit, da kein sichtbares Leadership-Hiring..."
7,Vertex Energy Transition,Energy,Denmark,PoC example,2,1,1,2,2,Executive hiring need in a high-transformation...,guter_fit,Guter Fit durch vorhandene Kontinuität und Wac...,hoher_fit,Hoher Fit durch starkes Leadership-Hiring-Sign...


In [845]:
fit_order = {
    "kein_fit": 0,
    "mindestfit": 1,
    "guter_fit": 2,
    "hoher_fit": 3
}

external_companies_poc_sorted = external_companies_poc.copy()

external_companies_poc_sorted["fit_longterm_order"] = external_companies_poc_sorted["fit_longterm_core_client"].map(fit_order)
external_companies_poc_sorted["fit_leadership_order"] = external_companies_poc_sorted["fit_leadership_search_client"].map(fit_order)

external_companies_poc_sorted.sort_values(
    by=["fit_longterm_order", "fit_leadership_order", "company_name"],
    ascending=[False, False, True]
)[
    [
        "company_name",
        "industry",
        "country",
        "fit_longterm_core_client",
        "fit_leadership_search_client",
        "fit_reason_longterm",
        "fit_reason_leadership"
    ]
]

,company_name,industry,country,fit_longterm_core_client,fit_leadership_search_client,fit_reason_longterm,fit_reason_leadership
0,AlphaTech Systems,Technology,Germany,hoher_fit,hoher_fit,"Hoher Fit durch starke Hiring-Kontinuität, sta...",Hoher Fit durch starkes Leadership-Hiring-Sign...
4,BluePeak Retail Europe,Retail,Netherlands,hoher_fit,kein_fit,"Hoher Fit durch starke Hiring-Kontinuität, sta...","Kein Fit, da kein sichtbares Leadership-Hiring..."
2,Crestline Health Solutions,Healthcare,Switzerland,guter_fit,hoher_fit,Guter Fit durch vorhandene Kontinuität und Wac...,Hoher Fit durch starkes Leadership-Hiring-Sign...
5,NovaMobility Ventures,Mobility,Germany,guter_fit,hoher_fit,Guter Fit durch vorhandene Kontinuität und Wac...,Hoher Fit durch starkes Leadership-Hiring-Sign...
7,Vertex Energy Transition,Energy,Denmark,guter_fit,hoher_fit,Guter Fit durch vorhandene Kontinuität und Wac...,Hoher Fit durch starkes Leadership-Hiring-Sign...
1,NordIndustrial Group,Industrial Manufacturing,Germany,guter_fit,guter_fit,Guter Fit durch vorhandene Kontinuität und Wac...,Guter Fit durch erkennbaren Leadership-Bedarf ...
3,Meridian Finance Advisory,Financial Services,Germany,kein_fit,guter_fit,"Kein Fit, da Kontinuität oder Wachstum nicht a...",Guter Fit durch erkennbaren Leadership-Bedarf ...
6,TradCore Services,Business Services,Austria,kein_fit,kein_fit,"Kein Fit, da Kontinuität oder Wachstum nicht a...","Kein Fit, da kein sichtbares Leadership-Hiring..."


### Erste Einordnung der PoC-Ergebnisse

Die Anwendung der segmentbezogenen Regelheuristiken auf die externe Beispiel-Unternehmensliste zeigt, dass sich die zuvor aus den internen Debitorsegmenten abgeleitete Logik grundsätzlich in eine einfache externe Ähnlichkeitsbewertung überführen lässt.

Für das Segment `langfristige Kernkunden` erhalten insbesondere solche Unternehmen höhere Fit-Bewertungen, die zugleich Kontinuität im Hiring und erkennbare Wachstumssignale aufweisen. Zusätzliche Signale wie Leadership-Hiring, Transformation oder Managementwechsel wirken dabei als strategische Verstärkung.

Für das Segment `führungsorientierte Suchkunden` ist vor allem ein sichtbares Leadership-Hiring-Signal ausschlaggebend. Die Passung steigt, wenn dieses Signal gemeinsam mit Transformationsdynamik oder Managementwechseln auftritt.

Die PoC-Ergebnisse zeigen damit, dass die interne Segmentlogik in eine nachvollziehbare externe Heuristik übersetzt werden kann. Gleichzeitig bleibt die Bewertung bewusst einfach und explorativ. Sie dient nicht als validiertes externes Scoring, sondern als methodische Demonstration, wie aus internen Kundenmustern erste Lookalike-Hypothesen für potenzielle Zielunternehmen abgeleitet werden können.

### Methodische Grenzen des heuristischen Lookalike-PoC

Der hier umgesetzte Lookalike-PoC besitzt bewusst einen heuristischen Charakter. Die verwendeten Signale sind keine validierten Prädiktoren für tatsächlichen Vertriebserfolg, sondern interpretative Proxy-Variablen, die strukturelle Ähnlichkeiten zu attraktiven internen Kundensegmenten annähern sollen.

Daraus ergeben sich mehrere Einschränkungen:

- Die externe Bewertung basiert nicht auf beobachtetem Abschluss- oder Mandatserfolg.
- Die Signalwerte wurden im PoC manuell beziehungsweise exemplarisch gesetzt und nicht aus einer produktiven externen Datenpipeline gewonnen.
- Die Heuristik vereinfacht komplexe Marktrealitäten auf wenige ordinale Signale und feste Schwellenwerte.
- Die Übertragbarkeit interner Segmentmuster auf externe Unternehmen ist konzeptionell plausibel, aber empirisch nicht validiert.
- Der PoC dient daher nicht der exakten Vorhersage, sondern der strukturierten Priorisierung und Hypothesenbildung für eine datenunterstützte Akquise.

Trotz dieser Einschränkungen stärkt der PoC die Gesamtlogik des Projekts, weil er den Übergang von interner Segmentierung zu externer Nutzbarmachung methodisch sichtbar macht.

## Reflexion von Anonymisierung, Bias, Limitationen und Übertragbarkeit

Der vorliegende Analyseansatz verfolgt das Ziel, aus internen historischen Sales-Daten strukturelle Kundentypen abzuleiten und diese in eine einfache externe Lookalike-Logik zu überführen. Damit ein solcher Ansatz methodisch belastbar eingeordnet werden kann, müssen neben den Ergebnissen auch die Grenzen der Datenbasis, der getroffenen Modellierungsentscheidungen und der externen Übertragbarkeit explizit reflektiert werden.

Im Folgenden werden deshalb vier Perspektiven zusammengeführt:

1. Anonymisierung und Datenschutzlogik
2. Datenqualitätsgrenzen der internen Analysebasis
3. Methodische Bias- und Modellgrenzen der Segmentierung
4. Grenzen des externen heuristischen Lookalike-Transfers

### 1. Anonymisierung und Datenschutzlogik

Die Analyse basiert nicht auf einer unveränderten Rohdatenextraktion, sondern auf einer gezielt vorbereiteten, datenschutzkonformen Analysebasis. Direkt identifizierende und potenziell re-identifizierende Felder wurden vor der eigentlichen Analyse entfernt oder aus der Modellierungslogik ausgeschlossen. Debitoren wurden konsistent anonymisiert, sodass die Segmentierung nicht auf konkreten Kundennamen, sondern ausschließlich auf aggregierten Strukturmerkmalen beruht.

Zusätzlich wurden monetäre Variablen nicht als Originalwerte weiterverarbeitet, sondern relativ zum jeweiligen Maximalwert auf 1.000 skaliert und gerundet. Dadurch bleibt die interne Größenrelation für analytische Zwecke grundsätzlich erhalten, ohne dass vertrauliche absolute Umsatzinformationen offengelegt werden. Diese Entscheidung erhöht den Datenschutz, führt jedoch zugleich dazu, dass absolute wirtschaftliche Größenordnungen nicht mehr direkt interpretierbar sind.

Die Anonymisierung verbessert somit die Vertraulichkeit der Analyse deutlich, verändert aber zugleich die direkte Lesbarkeit einzelner Variablen. Der Ansatz ist daher bewusst als analytisch nutzbare, aber nicht vollständig rohdatentreue Repräsentation der ursprünglichen Geschäftsdaten zu verstehen.

### 2. Datenqualitätsgrenzen der internen Analysebasis

Die interne Analysebasis umfasst historische Fälle aus dem Zeitraum 2010 bis 2025 und ist auf den fachlich relevanten Scope `Personalberatung` begrenzt. Diese Fokussierung verbessert die inhaltliche Homogenität der Datenbasis, reduziert jedoch zugleich die Breite möglicher Geschäftskontexte. Das Modell beschreibt daher nicht alle Formen historischer Kundenbeziehungen, sondern nur den bewusst gewählten fachlichen Ausschnitt.

Hinzu kommt, dass nicht alle Felder über den gesamten Zeitraum hinweg gleich konsistent gepflegt wurden. Einzelne Variablen mussten aufgrund begrenzter Belastbarkeit ausgeschlossen oder nur eingeschränkt berücksichtigt werden. Das Feld `Gesamthonorar` wurde beispielsweise nicht als robuste Analysevariable verwendet, da die Pflegequalität nicht ausreichend konsistent war. Auch `OTE` wurde nicht als allgemeines Kernmerkmal behandelt, sondern nur dort berücksichtigt, wo das Merkmal fachlich anwendbar war, nämlich bei tatsächlichen Placement-Fällen.

Ein weiterer methodischer Aspekt betrifft die zeitliche Vergleichbarkeit. Der lange Analysezeitraum erhöht zwar die Datenbasis, kann aber auch strukturelle Veränderungen in Markt, Geschäftsmodell, Erfassungslogik oder Vertriebsrealität überdecken. Die analysierten Debitorprofile stellen daher keine vollständig zeitstabile Abbildung eines unveränderten Marktes dar, sondern verdichten historische Beziehungen über einen langen und potenziell heterogenen Zeitraum.

### 3. Methodische Bias- und Modellgrenzen der Segmentierung

Die identifizierten Kundensegmente sind nicht als objektiv gegebene Klassen zu verstehen, sondern als Ergebnis konkreter methodischer Entscheidungen. Dazu gehören insbesondere die Wahl der Analyseeinheit Debitor, die Aggregationslogik, die Auswahl und Reduktion der Features, die Skalierung der Variablen sowie die Wahl des Clustering-Verfahrens.

Mit K-Means wurde bewusst ein einfaches und gut nachvollziehbares Baseline-Verfahren gewählt. Diese Entscheidung erhöht die Transparenz, bringt aber zugleich bekannte Einschränkungen mit sich. K-Means reagiert sensibel auf Skalierung, Merkmalsauswahl, Clusterzahl und geometrische Datenstruktur. Die entstehenden Cluster sind daher immer auch ein Produkt der Modellierungslogik und nicht nur ein direkter Spiegel der zugrunde liegenden Realität.

Die Robustheitsprüfung hat genau diese Problematik sichtbar gemacht. Die metrisch stärkste Referenzlösung mit `k = 2` erwies sich inhaltlich als stark durch Größen-, Aktivitäts- und Intensitätsmerkmale geprägt. Deshalb wurde mit `C_Relational` bewusst eine fachlich balanciertere, aber nicht metrisch dominante Variante als Hauptlösung gewählt. Diese Entscheidung ist methodisch begründbar und business-seitig plausibel, bleibt aber dennoch interpretativ. Die Segmentlösung ist somit nicht als endgültige Wahrheit zu verstehen, sondern als reflektierte analytische Annäherung an strukturelle Kundentypen.

### 4. Grenzen des externen heuristischen Lookalike-Transfers

Der externe Lookalike-Transfer wurde bewusst nicht als validiertes Scoring-Modell, sondern als heuristischer Proof of Concept angelegt. Die Übertragung interner Segmentmuster auf externe Unternehmen erfolgt daher nicht über beobachtete Erfolgsdaten, sondern über Proxy-Signale, die strukturelle Ähnlichkeiten annähern sollen.

Diese Proxy-Signale, etwa zu Leadership Hiring, Hiring Continuity, Wachstum, Transformation oder Managementwechsel, sind konzeptionell plausibel, stellen aber keine empirisch validierten Prädiktoren für tatsächlichen Akquise-Erfolg dar. Hinzu kommt, dass im aktuellen PoC keine produktive externe Datenpipeline verwendet wurde, sondern eine kleine exemplarische Unternehmensliste mit manuell gesetzten Signalwerten. Dadurch eignet sich der Ansatz gut zur methodischen Demonstration, aber noch nicht zur belastbaren externen Priorisierung im operativen Vertrieb.

Auch die Übertragbarkeit interner Muster auf externe Zielunternehmen bleibt grundsätzlich begrenzt. Historisch erfolgreiche Kundenbeziehungen können auf Kontextfaktoren beruhen, die in der Analysebasis nicht vollständig sichtbar sind, etwa Netzwerkqualität, Mandatsgeschichte, individuelle Beziehungskonstellationen oder vertriebliche Zufälligkeiten. Der PoC zeigt damit sinnvoll eine Richtung auf, ersetzt aber keine empirische Validierung mit realen externen Daten und beobachteten Vertriebsergebnissen.

### Zusammenfassende Einordnung

Insgesamt zeigt das Projekt, dass ein realer Executive-Search-Business-Case in einen nachvollziehbaren datengetriebenen Analyseprozess überführt werden kann. Gleichzeitig machen die reflektierten Grenzen deutlich, dass weder die interne Segmentierung noch der externe Lookalike-Transfer als endgültige oder vollständig objektive Wahrheit interpretiert werden dürfen.

Die Stärke des Ansatzes liegt daher weniger in exakter Vorhersage als in strukturierter Verdichtung, nachvollziehbarer Musterbildung und datenunterstützter Hypothesenbildung. Gerade in dieser reflektierten Form erfüllt der Ansatz sowohl einen praktischen Business-Zweck als auch die Anforderungen einer akademisch sauber eingeordneten Prüfungsleistung.

## Dokumentation der Datenquelle und Analysebasis

Die vorliegende Analyse basiert auf einer internen operativen Datenbasis aus dem Executive-Search-Kontext. Ausgangspunkt war kein bereits wissenschaftlich vorbereiteter Forschungsdatensatz, sondern ein historischer Arbeitsbestand, der für Analysezwecke zunächst strukturiert geprüft, fachlich reduziert und methodisch transformiert werden musste.

Ziel der Datenvorbereitung war es, aus einer operativen Rohdatenbasis eine nachvollziehbare, reproduzierbare und datenschutzkonforme Analysebasis zu erzeugen, die für die Untersuchung struktureller Kundentypen auf Debitor-Ebene geeignet ist. Die finale Analysebasis ist daher nicht als unveränderte Rohdatenkopie zu verstehen, sondern als dokumentiert vorbereitete Arbeitsgrundlage für das Assignment.

### Herkunft der Datenbasis

Die Daten stammen aus einem internen historischen Export der vertriebs- und abrechnungsnahen Geschäftsvorgänge im Executive-Search-Umfeld. Die zugrunde liegenden Einheiten der Rohdaten sind einzelne operative Fälle beziehungsweise Abrechnungs- und Umsatzvorgänge, die im Projekt jedoch nicht als finale Analyseeinheit verwendet werden. Stattdessen dienen sie als Rohmaterial für die spätere Aggregation auf Debitor-Ebene.

Die operative Ausgangsdatei wurde zunächst technisch eingelesen, strukturell geprüft und anschließend in eine bereinigte Analysebasis überführt. Diese Transformation war notwendig, weil die ursprünglichen Daten sowohl fachlich heterogene Umsatzarten als auch operative, sensitive oder nur eingeschränkt belastbare Felder enthielten, die für die eigentliche Analysefrage nicht direkt geeignet waren.

### Herkunft und Rolle zentraler Felder

Die verwendeten Analysemerkmale stammen aus der operativen Ursprungsdatei, wurden jedoch nicht unverändert übernommen. Vielmehr wurde zwischen direkt nutzbaren, transformierten, aggregierten und ausgeschlossenen Feldern unterschieden.

Direkt oder in leicht bereinigter Form nutzbar waren insbesondere Merkmale, die Aktivität, zeitliche Einordnung, Placement-Bezug, Führungsrelevanz sowie strukturelle Eigenschaften von Mandaten oder Kundenbeziehungen beschreiben. Dazu zählen etwa Informationen zu Debitor, Datum, Umsatzart, Vertragskontext, Branche, Jobkategorie, Placement und Führungsposition.

Transformiert wurden vor allem monetäre Variablen, die nicht als absolute Originalwerte weiterverarbeitet, sondern relativ skaliert wurden. Aggregiert wurden die Rohinformationen anschließend auf Debitor-Ebene, sodass aus einzelnen Vorgängen strukturierte Kundenprofile entstehen konnten.

Ausgeschlossen wurden Felder, die entweder nicht relevant, nicht konsistent genug gepflegt oder aus Datenschutz- und Re-Identifikationssicht problematisch waren. Dazu zählen insbesondere direkt identifizierende Referenzen, operative Beschreibungsfelder sowie einzelne fachlich nicht belastbare Variablen wie `Gesamthonorar`.

### Begründung des Analysezeitraums

Die Analysebasis umfasst historische Fälle aus dem Zeitraum 2010 bis 2025. Der gewählte Zeitraum wurde bewusst breit gehalten, um ausreichend viele historische Kundenbeziehungen für eine strukturorientierte Debitoranalyse verfügbar zu machen. Gerade für die Bildung aggregierter Kundenprofile ist eine gewisse historische Tiefe hilfreich, da einzelne Debitoren sonst nur sehr punktuell sichtbar wären.

Die Nutzung eines langen Zeitraums bringt jedoch nicht nur Vorteile, sondern auch methodische Grenzen mit sich. Marktbedingungen, Geschäftslogiken, Erfassungspraktiken und vertriebliche Kontexte können sich über die Jahre verändert haben. Die Analyse verdichtet daher historische Beziehungen über einen längeren Zeitraum und bildet nicht ausschließlich eine aktuelle Marktlogik ab. Der Zeitraum ist damit als pragmatischer Kompromiss zwischen Datenverfügbarkeit, Fallzahl und historischer Heterogenität zu verstehen.

### Begründung der fachlichen Scope-Reduktion

Für die eigentliche Analyse wurde die Datenbasis bewusst auf den fachlich relevanten Scope `Personalberatung` reduziert. Ziel dieser Entscheidung war es, eine möglichst homogene und inhaltlich vergleichbare Grundlage für die Debitorprofilierung und Segmentierung zu schaffen.

Andere Umsatzarten wie `Training` oder `Sonstiges` wurden ausgeschlossen, weil sie nicht denselben fachlichen Kern abbilden und die Struktur der Debitorprofile verzerren könnten. Die Segmentierung soll nicht allgemeine Geschäftsaktivität messen, sondern strukturelle Muster jener Kundenbeziehungen sichtbar machen, die tatsächlich dem Executive-Search- und Personalberatungs-Kontext zuzuordnen sind.

Die Scope-Reduktion ist deshalb nicht nur ein technischer Filter, sondern eine zentrale methodische Entscheidung zur Erhöhung der fachlichen Aussagekraft der Analyse.

### Zusammenfassende Einordnung der Analysebasis

Insgesamt basiert das Projekt auf einer realen, operativen und zunächst nicht analysefertigen Datenbasis, die methodisch in eine fokussierte Debitor-Analysebasis überführt wurde. Die Dokumentation von Herkunft, Zeitraum, Feldrolle und Scope-Reduktion ist dabei zentral, um die Reproduzierbarkeit und Nachvollziehbarkeit des Projekts sicherzustellen.

Die Analysebasis ist damit weder als vollständige Rohdatenabbildung noch als beliebiger Datenausschnitt zu verstehen, sondern als fachlich begründete, dokumentierte und für die konkrete Forschungsfrage vorbereitete Arbeitsgrundlage.